# Chapter 40 — Composing Agent Operations

**Companion to *Pi Agents*** · [`Pi Agents` chapter 40](https://programmer.ie/books/pi/40-chapter/)

| | |
|---|---|
| Chapter | `40-chapter.md` · weight 40 · `/books/pi/40-chapter/` |
| Notebook | `notebooks/pi/40-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

What is the exact request cost of each route, and does a check placed *after*
the model stop an invented citation before the next call is paid for?

The chapter's point: an agent pipeline is a **composition of typed operations**.
The escalation-only-if-needed route costs exactly one assessment and one summary
on the cheap path, and the agent's escalation is bounded. A citation check placed
after the assessment stops an invented citation **before the summary call**.


## What this notebook establishes

- The **deterministic parts need no model at all** — retrieval and the citation check run without a provider.
- The **cheap path** is one assessment call and one summary step — **no agent**, exactly 2 requests.
- An **invented citation stops the pipeline** before anything downstream runs — the summary step is not paid for.
- A **truthful citation on the escalation path** passes the second check.
- **Escalation only pays for the agent when the verdict is insufficient**.
- The escalation agent is **bounded**, so a model that keeps searching cannot run away.
- An escalation **cut off before it found anything fails loudly** instead of re-deciding on no new evidence.

## What this notebook does **not** establish

- **Request counts are exact for the scripted control flow only** (`ch40-lim2`) — a real model may take a different path.
- **No real model is called** — this measures the pipeline's control flow, not model ability.
- **The citation check is a string match against evidence**, not a semantic one.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the chapter's `checkClaim` composition against the scripted provider.
* **Evidence scope:** `in_process_runtime`. The pipeline's control flow and bounds are the chapter's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(40))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/40-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the deterministic parts need no model

In [3]:
PIPE = pinb.driver_source("ch40-composition.ts")
pipe = pinb.driver(PIPE, label="ch40-composition", timeout=180)
d = pipe["deterministic"]
print(pinb.md_table(
    ["check", "value"],
    [["retrieve found the evidence", d["retrieved"]],
     ["an invented citation fails the check", d["citationOk"]]],
))

| check | value |
|---|---|
| retrieve found the evidence | ['The release shipped on 4 October.'] |
| an invented citation fails the check | False |


In [4]:
pinb.show_checks([
    pinb.check("retrieval returns the matching evidence",
               d["retrieved"] == ["The release shipped on 4 October."], "retrieved"),
    pinb.check("an invented citation is not real",
               d["citationOk"] is False, "not real"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------  ---------
PASS  retrieval returns the matching evidence  retrieved
PASS  an invented citation is not real  not real

2/2 assertions held.


## The cheap path: no agent, two requests

In [5]:
c = pipe["cheapPath"]
print(pinb.md_table(
    ["check", "value"],
    [["route", c["route"]],
     ["request count", c["callCount"]]],
))

| check | value |
|---|---|
| route | single call |
| request count | 2 |


In [6]:
pinb.show_checks([
    pinb.check("the cheap path is a single assessment and a summary",
               c["route"] == "single call" and c["callCount"] == 2, "2 requests, no agent"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------  --------------------
PASS  the cheap path is a single assessment and a summary  2 requests, no agent

1/1 assertions held.


## A check after the model stops an invented citation

In [7]:
i, et, ec = pipe["inventedCitation"], pipe["escalationTruthful"], pipe["escalationCost"]
print(pinb.md_table(
    ["case", "result", "requests"],
    [["invented citation", "rejected" if i["rejected"] else "accepted", i["callCount"]],
     ["escalation, truthful citation", f"route={et['route']}", et["callCount"]],
     ["escalation, supported verdict", f"route={ec['route']}, verdict={ec['verdict']}", ec["callCount"]]],
))

| case | result | requests |
|---|---|---|
| invented citation | rejected | 1 |
| escalation, truthful citation | route=agent escalation | 5 |
| escalation, supported verdict | route=agent escalation, verdict=supported | 5 |


In [8]:
pinb.show_checks([
    pinb.check("an invented citation stops before the summary is paid for",
               i["rejected"] and i["callCount"] == 1, "1 request, stopped"),
    pinb.check("a truthful citation on the escalation path passes",
               et["route"] == "agent escalation", "escalated and passed"),
    pinb.check("escalation only pays for the agent when the verdict is insufficient",
               ec["route"] == "agent escalation" and ec["verdict"] == "supported", "agent used once"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------------------  --------------------
PASS  an invented citation stops before the summary is paid for  1 request, stopped
PASS  a truthful citation on the escalation path passes  escalated and passed
PASS  escalation only pays for the agent when the verdict is insufficient  agent used once

3/3 assertions held.


## The chapter's own tests

All eight pipeline tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [9]:
PATTERNS = 'ch40-composition/pipeline.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch40-composition/pipeline.test.ts', show="escalation")

$ node --test ch40-composition/pipeline.test.ts
  PASS  8 passed, 0 failed, 0 skipped  in 400 ms

  tests matching 'escalation':
    ok   an invented citation on the escalation path is caught too
    ok   escalation: only an insufficient verdict pays for the agent
    ok   the escalation agent is bounded, so a model that keeps searching cannot run away
    ok   an escalation cut off before it found anything fails loudly instead of re-deciding on no new evidence


## Your turn: predict, then run

**Predict first.** The cheap path costs 2 requests. What does the escalation path
cost when the verdict is `insufficient` and the agent finds evidence on its
first turn?

**Then change one input.** The citation check matches evidence **by content
overlap**. What happens with a citation that is a paraphrase of the evidence —
passes or fails?

**Predict the boundary.** The escalation agent is bounded by `ESCALATION_TURNS`.
What happens if the model keeps searching to the bound and never finds
evidence?

In [10]:
print("Predictions:")
print("  1. Escalation cost: 1 assess + 1 agent turn + 1 assess + 1 summary = 4.")
print("  2. A paraphrase: fails the string-based check (semantic checking is not here).")
print("  3. Bound reached with no evidence: the pipeline fails loudly.")
print()
print("Observed above:")
print(f"  deterministic: retrieved={d['retrieved']}, citationOk={d['citationOk']}")
print(f"  cheap path: route={c['route']}, requests={c['callCount']}")
print(f"  invented citation: rejected={i['rejected']}, requests={i['callCount']}")
print(f"  escalation truthful: route={et['route']}, requests={et['callCount']}")
print()
assert c["route"] == "single call" and c["callCount"] == 2
assert i["rejected"] and i["callCount"] == 1
assert et["route"] == "agent escalation"
print("held: cheap path costs 2; an invented citation stops before the summary; escalation is bounded")

Predictions:
  1. Escalation cost: 1 assess + 1 agent turn + 1 assess + 1 summary = 4.
  2. A paraphrase: fails the string-based check (semantic checking is not here).
  3. Bound reached with no evidence: the pipeline fails loudly.

Observed above:
  deterministic: retrieved=['The release shipped on 4 October.'], citationOk=False
  cheap path: route=single call, requests=2
  invented citation: rejected=True, requests=1
  escalation truthful: route=agent escalation, requests=5

held: cheap path costs 2; an invented citation stops before the summary; escalation is bounded


## Interpretation

Composition is a set of decisions with known costs, and the chapter shows each:

| Route | Request cost | What stops it |
|---|---|---|
| **Cheap path** | 1 assessment + 1 summary = 2 | A `supported` verdict with real citations |
| **Invented citation** | 1 assessment | The check **after** the model, before the summary |
| **Agent escalation** | 1 + agent turns + 1 + 1 | A verifiable verdict; bounded by `ESCALATION_TURNS` |
| **Cut off** | 1 + agent turns | Fails loudly, never re-decides on no evidence |

The practical rules:
1. **Put the cheap check before the expensive call** — the citation check stops the summary request.
2. **Escalate only when needed** — an `insufficient` verdict is the only thing that pays for the agent.
3. **Bound the escalation** — a model that keeps searching cannot run away.
4. **Fail loudly on no evidence** — do not re-decide on nothing.
5. **Count requests** — the pipeline's cost is exact for the scripted control flow; a real model may diverge.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the chapter's `checkClaim` composition against the scripted provider.

**Evidence scope:** `in_process_runtime`. The pipeline's control flow and bounds are the chapter's.

### What was read or run

- **Ran** `drivers/ch40-composition.ts`: deterministic parts, cheap path, invented citation, escalation truthful, escalation cost.
- **Ran** `ch40-composition/pipeline.test.ts` (8 tests).
- **Read** `examples/evidence.json`, chapter 40 rows (8 claims).

### Limitations

- **Request counts are exact for the scripted control flow only** (`ch40-lim2`) — a real model may take a different path.
- **No real model is called** — this measures the pipeline's control flow, not model ability.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
